# Using Ollama (Local Models) as Your Model Provider

Ollama lets you run large language models **locally** on your own machine — no API keys,
no cloud, no data leaving your network. This notebook shows how to use locally-hosted
models like Llama 3, Mistral, and Gemma with MemoRizz.

**What you'll learn:**
1. Setting up Ollama and pulling models
2. Connecting MemoRizz to your local Ollama server
3. Tool calling with local models
4. Streaming responses
5. Performance tips for local inference

> **Prerequisites:**
> - Install Ollama: [ollama.com/download](https://ollama.com/download)
> - Pull a model: `ollama pull llama3.1`
> - Ensure the Ollama server is running (starts automatically after install)

In [ ]:
%pip install -qU memorizz ollama

---
## Verify Ollama is Running

Before proceeding, make sure Ollama is running and has at least one model pulled.

In [ ]:
import ollama

# List available models
models = ollama.list()
print("Available models:")
for model in models.get("models", []):
    name = model.get("name", model.get("model", "unknown"))
    size = model.get("size", 0)
    print(f"  - {name} ({size / 1e9:.1f} GB)")

if not models.get("models"):
    print("\nNo models found! Run: ollama pull llama3.1")

---
## Method 1: Config Dict (Recommended)

Set `"provider"` to `"ollama"` and specify the model name (must be pulled locally).

In [ ]:
from memorizz.memagent.builders import MemAgentBuilder

agent = (
    MemAgentBuilder()
    .with_instruction("You are a helpful assistant running locally.")
    .with_llm_config({
        "provider": "ollama",
        "model": "llama3.1",
    })
    .build()
)

response = agent.run("What are the advantages of running AI models locally?")
print(response)

## Method 2: Direct Provider Instance

Use this for custom host URLs, timeouts, or generation parameters.

In [ ]:
from memorizz.llms import OllamaLLM
from memorizz.memagent.builders import MemAgentBuilder

llm = OllamaLLM(
    model="llama3.1",
    host="http://localhost:11434",  # Default; change if Ollama runs elsewhere
    temperature=0.7,
    num_predict=2048,  # Max tokens to generate
    top_p=0.9,
)

agent = (
    MemAgentBuilder()
    .with_instruction("You are a code reviewer. Be constructive and specific.")
    .with_model(llm)
    .build()
)

response = agent.run("Review this code: `def add(a, b): return a + b`")
print(response)

---
## Connecting to a Remote Ollama Server

Ollama can run on a different machine (e.g. a GPU server). Just point to its address:

In [ ]:
# Example: connect to Ollama running on a remote GPU server
# (Update the host to your actual server address)

# remote_agent = (
#     MemAgentBuilder()
#     .with_instruction("You are a helpful assistant.")
#     .with_llm_config({
#         "provider": "ollama",
#         "model": "llama3.1:70b",
#         "host": "http://gpu-server.local:11434",
#     })
#     .build()
# )

print("Remote Ollama example (commented out — update host to use)")

---
## Tool Calling with Local Models

Many Ollama models support tool calling (Llama 3.1+, Qwen 2.5+, Mistral, etc.).
MemoRizz registers and calls tools the same way as with cloud providers.

In [ ]:
import math
from memorizz.memagent.builders import MemAgentBuilder


def circle_area(radius: float) -> float:
    """Calculate the area of a circle given its radius."""
    return round(math.pi * radius ** 2, 4)


def rectangle_area(width: float, height: float) -> float:
    """Calculate the area of a rectangle given width and height."""
    return round(width * height, 4)


agent = (
    MemAgentBuilder()
    .with_instruction("You are a geometry assistant. Always use tools for calculations.")
    .with_llm_config({"provider": "ollama", "model": "llama3.1"})
    .build()
)

agent.register_tool(circle_area)
agent.register_tool(rectangle_area)

response = agent.run("What is the area of a circle with radius 5?")
print(response)

---
## Streaming Responses

Streaming works the same as with other providers.

In [ ]:
for event in agent.run_stream("Explain recursion with a simple example."):
    if event.get("type") == "content":
        print(event["content"], end="", flush=True)
print()

---
## Popular Models to Try

| Model | Size | Best For | Pull Command |
|-------|------|----------|-------------|
| `llama3.1` | 8B | General purpose, tool calling | `ollama pull llama3.1` |
| `llama3.2` | 3B | Fast, lightweight tasks | `ollama pull llama3.2` |
| `llama3.3` | 70B | High-quality reasoning | `ollama pull llama3.3` |
| `mistral` | 7B | Fast general purpose | `ollama pull mistral` |
| `gemma3` | 9B/27B | Google's latest | `ollama pull gemma3` |
| `qwen3` | varies | Multilingual, tool calling | `ollama pull qwen3` |
| `deepseek-r1` | varies | Advanced reasoning | `ollama pull deepseek-r1` |
| `codellama` | 7B-34B | Code generation | `ollama pull codellama` |

> **Tip:** For tool calling, use `llama3.1` or newer. Older models may not support
> the structured tool-calling protocol.

## Config Reference

| Key | Type | Default | Description |
|-----|------|---------|-------------|
| `provider` | str | — | Must be `"ollama"` |
| `model` | str | `"llama3.1"` | Model name (must be pulled locally) |
| `host` | str | `OLLAMA_HOST` env or `localhost:11434` | Ollama server URL |
| `temperature` | float | None | Sampling temperature |
| `num_predict` | int | 4096 | Max tokens to generate |
| `top_p` | float | None | Nucleus sampling |
| `top_k` | int | None | Top-K sampling |
| `seed` | int | None | Random seed |
| `timeout` | float | None | Request timeout in seconds |

## Performance Tips

1. **GPU acceleration:** Ollama automatically uses your GPU if available. Check with `ollama ps`
2. **Model size vs quality:** 8B models run on most machines; 70B+ need 64GB+ RAM or a good GPU
3. **Keep-alive:** By default Ollama keeps models loaded for 5 minutes. For batch processing,
   set `keep_alive=-1` in the Ollama config to keep models loaded indefinitely
4. **Quantization:** Ollama serves quantized models by default (Q4_0). Specify a tag for
   different quantization: `ollama pull llama3.1:8b-instruct-q8_0`